# Notebook 2 — ESCO Occupation Standardization

**Project:** Kosovo Labor Market Demand Analysis 2025  
**Description:** This notebook matches translated English job titles against the EU ESCO v1.2 occupation dataset using a two-stage pipeline: fuzzy string matching followed by semantic similarity matching. Each job posting receives a standardized ESCO occupation label and a 4-digit ISCO-08 code.

**Input:** `superpuna_cleaned.csv` from Google Drive (produced by Notebook 1)  
**Output:** `superpuna_standardized.csv` saved to Google Drive  
**Next step:** Run `03_demand_analysis.ipynb`

**References:**
- ESCO v1.2: https://esco.ec.europa.eu
- ISCO-08: https://www.ilo.org/public/english/bureau/stat/isco/isco08/

---

## 1. Install and Import Libraries

In [ ]:
!pip install rapidfuzz sentence-transformers scikit-learn tqdm openpyxl -q

In [ ]:
# Standard library
import os

# Data
import pandas as pd
import numpy as np

# Matching
from rapidfuzz import process, fuzz
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm

# Colab utilities
from google.colab import files, drive

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

## 2. Constants

In [ ]:
# Matching thresholds
# Fuzzy: minimum token sort ratio score (0-100) to accept a match
# Semantic: minimum cosine similarity (0-1) to accept a match
# Raising these values increases precision but reduces coverage
FUZZY_THRESHOLD    = 85
SEMANTIC_THRESHOLD = 0.50

# Column names
TITLE_COL   = 'Titulli_i_vendit_te_punes'
VACANCY_COL = 'Numri_i_vendeve_te_lira'
COMPANY_COL = 'Kompania'
CITY_COL    = 'Vendi_i_punes'
SALARY_COL  = 'Paga'
DATE_COL    = 'Data_e_shpalljes'

# ISCO-08 major group labels (ILO, 2008)
ISCO_MAJOR_GROUPS = {
    '1': 'Managers',
    '2': 'Professionals',
    '3': 'Technicians and associate professionals',
    '4': 'Clerical support workers',
    '5': 'Service and sales workers',
    '6': 'Skilled agricultural, forestry and fishery workers',
    '7': 'Craft and related trades workers',
    '8': 'Plant and machine operators and assemblers',
    '9': 'Elementary occupations',
    '0': 'Armed forces occupations'
}

DRIVE_DIR = '/content/drive/MyDrive/kosovo_lm'

print('Constants defined.')

## 3. Load Cleaned Dataset from Google Drive

In [ ]:
drive.mount('/content/drive')

df = pd.read_csv(f'{DRIVE_DIR}/superpuna_cleaned.csv')
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')

saved = pd.read_csv(f'{DRIVE_DIR}/translation_map.csv')
translation_map = dict(zip(saved['albanian'], saved['english']))

print(f'Dataset loaded       : {len(df):,} rows')
print(f'Translation map loaded: {len(translation_map):,} titles')

## 4. Upload ESCO Occupation Dataset

In [ ]:
# Upload occupations_en.csv downloaded from:
# https://esco.ec.europa.eu/en/use-esco/download
# Version: v1.2.0 | Language: English | Format: CSV
print('Upload occupations_en.csv from the ESCO v1.2 download...')
uploaded = files.upload()
fname = list(uploaded.keys())[0]
esco  = pd.read_csv(fname)

required_cols = ['preferredLabel', 'altLabels', 'iscoGroup', 'conceptUri']
missing = [c for c in required_cols if c not in esco.columns]
if missing:
    print(f'Error: missing columns {missing}. Verify you uploaded occupations_en.csv.')
else:
    print(f'ESCO dataset loaded: {len(esco):,} occupations.')
    print(f'Columns verified: {required_cols}')

## 5. Build ESCO Lookup Table

In [ ]:
# Expand the ESCO dataset into a flat lookup table.
# Each occupation has one preferred label and multiple alternative labels.
# Expanding alternative labels into individual rows means synonyms
# like 'waiter', 'table server', 'food server' all map to the same
# ESCO occupation during matching.

esco_lookup_rows = []

for _, row in esco.iterrows():
    preferred_display = str(row['preferredLabel']).strip()
    preferred_lower   = preferred_display.lower()
    isco_code         = str(row.get('iscoGroup', '')).strip()
    concept_uri       = str(row.get('conceptUri', '')).strip()

    esco_lookup_rows.append({
        'label'          : preferred_lower,
        'preferred_label': preferred_display,
        'isco_group'     : isco_code,
        'concept_uri'    : concept_uri,
        'label_type'     : 'preferred'
    })

    alt = str(row.get('altLabels', ''))
    if alt and alt != 'nan':
        for alt_label in alt.split('\n'):
            alt_label = alt_label.strip().lower()
            if alt_label:
                esco_lookup_rows.append({
                    'label'          : alt_label,
                    'preferred_label': preferred_display,
                    'isco_group'     : isco_code,
                    'concept_uri'    : concept_uri,
                    'label_type'     : 'alternative'
                })

esco_lookup = pd.DataFrame(esco_lookup_rows).drop_duplicates(subset=['label'])

print('ESCO lookup table built.')
print(f'Preferred labels   : {(esco_lookup["label_type"]=="preferred").sum():,}')
print(f'Alternative labels : {(esco_lookup["label_type"]=="alternative").sum():,}')
print(f'Total searchable   : {len(esco_lookup):,}')

## 6. Stage 1 — Fuzzy String Matching

In [ ]:
# Fuzzy matching compares character similarity between strings.
# It handles minor spelling differences and word order variations
# but struggles with semantically similar titles that use different words
# (e.g. 'seller' vs 'salesperson'). Those cases go to Stage 2.

esco_labels_lower = esco_lookup['label'].str.lower().tolist()

def fuzzy_match_title(title, threshold=FUZZY_THRESHOLD):
    if not isinstance(title, str) or title.strip() == '':
        return None, 0
    result = process.extractOne(
        title.lower().strip(),
        esco_labels_lower,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=threshold
    )
    if result:
        return esco_lookup.iloc[result[2]], result[1]
    return None, 0


unique_english   = df['title_english'].dropna().unique().tolist()
fuzzy_results    = {}
unmatched_titles = []

print('Stage 1: Fuzzy matching...')
for title in tqdm(unique_english, desc='Fuzzy matching'):
    esco_row, score = fuzzy_match_title(title)
    if esco_row is not None:
        fuzzy_results[title] = {
            'esco_preferred_label': esco_row['preferred_label'],
            'isco_group'          : esco_row['isco_group'],
            'concept_uri'         : esco_row['concept_uri'],
            'match_score'         : round(score, 1),
            'match_method'        : 'fuzzy'
        }
    else:
        unmatched_titles.append(title)

print(f'Fuzzy matched  : {len(fuzzy_results):,} titles')
print(f'Unmatched      : {len(unmatched_titles):,} titles (proceeding to Stage 2)')

## 7. Stage 2 — Semantic Similarity Matching

In [ ]:
# Semantic matching encodes titles as dense vectors using a pre-trained
# sentence transformer model. Titles with similar meaning produce similar
# vectors, regardless of the specific words used. This catches cases where
# fuzzy matching fails due to vocabulary differences.
#
# Model: all-MiniLM-L6-v2 (Reimers & Gurevych, 2019)
# Similarity metric: cosine similarity

if unmatched_titles:
    print('Stage 2: Semantic matching...')
    print('Loading sentence transformer model (approx. 90MB on first run)...\n')

    model = SentenceTransformer('all-MiniLM-L6-v2')

    esco_preferred_only   = esco_lookup[esco_lookup['label_type'] == 'preferred'].copy()
    esco_preferred_labels = esco_preferred_only['label'].tolist()

    print(f'Encoding {len(esco_preferred_labels):,} ESCO preferred labels...')
    esco_embeddings = model.encode(esco_preferred_labels, batch_size=256,
                                    show_progress_bar=True, convert_to_numpy=True)

    print(f'Encoding {len(unmatched_titles):,} unmatched job titles...')
    query_embeddings = model.encode(unmatched_titles, batch_size=256,
                                     show_progress_bar=True, convert_to_numpy=True)

    similarities     = cosine_similarity(query_embeddings, esco_embeddings)
    semantic_results = {}

    for i, title in enumerate(unmatched_titles):
        best_idx   = np.argmax(similarities[i])
        best_score = similarities[i][best_idx]
        if best_score >= SEMANTIC_THRESHOLD:
            esco_row = esco_preferred_only.iloc[best_idx]
            semantic_results[title] = {
                'esco_preferred_label': esco_row['preferred_label'],
                'isco_group'          : esco_row['isco_group'],
                'concept_uri'         : esco_row['concept_uri'],
                'match_score'         : round(float(best_score) * 100, 1),
                'match_method'        : 'semantic'
            }

    print(f'Semantic matched : {len(semantic_results):,} additional titles')
    print(f'Still unmatched  : {len(unmatched_titles) - len(semantic_results):,} titles')
else:
    semantic_results = {}
    print('No unmatched titles — fuzzy matching covered all titles.')

## 8. Build Standardized Dataset

In [ ]:
all_matches = {**fuzzy_results, **semantic_results}

def get_match_field(title, field):
    if title in all_matches:
        return all_matches[title].get(field)
    return None

df['esco_occupation']  = df['title_english'].apply(lambda t: get_match_field(t, 'esco_preferred_label'))
df['isco_group']       = df['title_english'].apply(lambda t: get_match_field(t, 'isco_group'))
df['esco_concept_uri'] = df['title_english'].apply(lambda t: get_match_field(t, 'concept_uri'))
df['match_score']      = df['title_english'].apply(lambda t: get_match_field(t, 'match_score'))
df['match_method']     = df['title_english'].apply(lambda t: get_match_field(t, 'match_method'))
df['isco_major_group'] = df['isco_group'].astype(str).str[0].map(ISCO_MAJOR_GROUPS)

total    = len(df)
matched  = df['esco_occupation'].notna().sum()

print('Standardized dataset built.')
print(f'Total postings   : {total:,}')
print(f'Matched to ESCO  : {matched:,} ({matched/total*100:.1f}%)')
print(f'Unmatched        : {total - matched:,} ({(total-matched)/total*100:.1f}%)')

## 9. Manual ESCO Overrides

In [ ]:
# Some titles match the wrong ESCO occupation because ESCO does not contain
# an exact equivalent. These overrides correct known mismatches identified
# during manual review of the matching output.
# Format: 'english title' : ('correct esco occupation', 'isco-08 code')

manual_esco_fixes = {
    'Operations Manager'               : ('operations manager',              '1324'),
    'Manager for social networks'      : ('social media manager',            '2642'),
    'Worker'                           : ('manufacturing worker',             '8211'),
    'R&D Technician and IT Specialist' : ('research and development manager', '1223'),
    'Information Technology Assistant' : ('ICT user support technician',      '3514'),
    'Administrator'                    : ('administrative assistant',         '4110'),
}

fixed_count = 0
for idx, row in df.iterrows():
    if row['title_english'] in manual_esco_fixes:
        correct_label, correct_isco = manual_esco_fixes[row['title_english']]
        df.at[idx, 'esco_occupation'] = correct_label
        df.at[idx, 'isco_group']      = correct_isco
        df.at[idx, 'match_method']    = 'manual'
        df.at[idx, 'match_score']     = 100.0
        fixed_count += 1

df['isco_major_group'] = df['isco_group'].astype(str).str[0].map(ISCO_MAJOR_GROUPS)

print(f'Manual overrides applied: {fixed_count:,} rows corrected')
print(f'Final match method breakdown:')
print(df['match_method'].value_counts().to_string())

## 10. Final Matching Summary

In [ ]:
total    = len(df)
matched  = df['esco_occupation'].notna().sum()

print('=' * 50)
print('         FINAL MATCHING SUMMARY')
print('=' * 50)
print(f'  Total job postings   : {total:,}')
print(f'  Matched to ESCO      : {matched:,} ({matched/total*100:.1f}%)')
print(f'  Unmatched            : {total - matched:,} ({(total-matched)/total*100:.1f}%)')
print(f'  Fuzzy matches        : {(df["match_method"]=="fuzzy").sum():,}')
print(f'  Semantic matches     : {(df["match_method"]=="semantic").sum():,}')
print(f'  Manual overrides     : {(df["match_method"]=="manual").sum():,}')
print(f'  Average match score  : {df["match_score"].mean():.1f}')
print('=' * 50)
print('\nDistribution by ISCO Major Group:')
print(df.groupby('isco_major_group')['isco_group'].count()
        .sort_values(ascending=False).to_string())

## 11. Save Standardized Dataset to Google Drive

In [ ]:
output_path = f'{DRIVE_DIR}/superpuna_standardized.csv'
df.to_csv(output_path, index=False)

print('Standardized dataset saved successfully.')
print(f'Path     : {output_path}')
print(f'Rows     : {len(df):,}')
print(f'Columns  : {list(df.columns)}')
print('\nNext step: open 03_demand_analysis.ipynb')